# 9주차: CNN Layer Shape 확인과 Batch Size/Parameter 분석

- 입력 shape: `(N, 1, 28, 28)`
- 목표: CNN 각 층의 출력 shape를 계산하고, 배치 크기에 따른 출력 shape와 파라미터 수를 확인한다.
- 추가로, `Linear` 입력 크기와 영상 크기가 맞지 않는 경우를 실험한다.


In [1]:
import pandas as pd
import torch
from torch import nn

print('PyTorch version:', torch.__version__)
torch.manual_seed(42)

x = torch.randn(8, 1, 28, 28)
cnn = nn.Sequential(
    nn.Conv2d(1, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(64, 128, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2, padding=1),
    nn.Flatten(start_dim=1),
    nn.Linear(2048, 625),
    nn.ReLU(),
    nn.Dropout(),
    nn.Linear(625, 10),
)

print('입력 shape:', tuple(x.shape))
print('모델 구조:')
print(cnn)


PyTorch version: 2.15.0+cpu
입력 shape: (8, 1, 28, 28)
모델 구조:
Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=1, dilation=1, ceil_mode=False)
  (9): Flatten(start_dim=1, end_dim=-1)
  (10): Linear(in_features=2048, out_features=625, bias=True)
  (11): ReLU()
  (12): Dropout(p=0.5, inplace=False)
  (13): Linear(in_features=625, out_features=10, bias=True)
)


## 1. CNN 각 층의 shape 확인하기

아래 CNN을 실행하기 전에 각 층의 예상 출력 shape를 적고, 실제 출력 shape를 비교한다.

1. `Conv2d(1,32,3,padding=1)` → `ReLU()` → `MaxPool2d(2)`
2. `Conv2d(32,64,3,padding=1)` → `ReLU()` → `MaxPool2d(2)`
3. `Conv2d(64,128,3,padding=1)` → `ReLU()` → `MaxPool2d(2,padding=1)`
4. `Flatten(start_dim=1)` → `Linear(2048,625)` → `ReLU()`
5. `Dropout()` → `Linear(625,10)`


In [2]:
expected_shapes = [
    ('Conv2d(1,32,3,padding=1)', '(N, 32, 28, 28)'),
    ('ReLU()', '(N, 32, 28, 28)'),
    ('MaxPool2d(2)', '(N, 32, 14, 14)'),
    ('Conv2d(32,64,3,padding=1)', '(N, 64, 14, 14)'),
    ('ReLU()', '(N, 64, 14, 14)'),
    ('MaxPool2d(2)', '(N, 64, 7, 7)'),
    ('Conv2d(64,128,3,padding=1)', '(N, 128, 7, 7)'),
    ('ReLU()', '(N, 128, 7, 7)'),
    ('MaxPool2d(2,padding=1)', '(N, 128, 4, 4)'),
    ('Flatten(start_dim=1)', '(N, 2048)'),
    ('Linear(2048,625)', '(N, 625)'),
    ('ReLU()', '(N, 625)'),
    ('Dropout()', '(N, 625)'),
    ('Linear(625,10)', '(N, 10)'),
]

records = []
tmp = x
for layer, (name, expected) in zip(cnn, expected_shapes):
    tmp = layer(tmp)
    records.append({
        'Layer': name,
        '예상 output shape': expected,
        '실제 output shape': str(tuple(tmp.shape)),
    })

layer_df = pd.DataFrame(records)
display(layer_df)
print('최종 출력 shape:', tuple(tmp.shape))
assert tuple(tmp.shape) == (8, 10)


,Layer,예상 output shape,실제 output shape
0,"Conv2d(1,32,3,padding=1)","(N, 32, 28, 28)","(8, 32, 28, 28)"
1,ReLU(),"(N, 32, 28, 28)","(8, 32, 28, 28)"
2,MaxPool2d(2),"(N, 32, 14, 14)","(8, 32, 14, 14)"
3,"Conv2d(32,64,3,padding=1)","(N, 64, 14, 14)","(8, 64, 14, 14)"
4,ReLU(),"(N, 64, 14, 14)","(8, 64, 14, 14)"
5,MaxPool2d(2),"(N, 64, 7, 7)","(8, 64, 7, 7)"
6,"Conv2d(64,128,3,padding=1)","(N, 128, 7, 7)","(8, 128, 7, 7)"
7,ReLU(),"(N, 128, 7, 7)","(8, 128, 7, 7)"
8,"MaxPool2d(2,padding=1)","(N, 128, 4, 4)","(8, 128, 4, 4)"
9,Flatten(start_dim=1),"(N, 2048)","(8, 2048)"


최종 출력 shape: (8, 10)


### 해석

- `Conv2d`는 입력 채널 수를 유지하면서 출력 채널 수만큼의 특징맵을 생성한다.
- `padding=1`이므로 공간 크기는 유지되며, `MaxPool2d(2)`는 절반으로 줄어든다.
- 마지막 `MaxPool2d(2,padding=1)`은 `7x7` 입력에 패딩을 적용해 `4x4`가 된다.
- `Flatten` 이후 `128 * 4 * 4 = 2048`이므로 `Linear(2048,625)` 입력 크기가 맞는다.
- 따라서 최종 출력은 `(N, 10)`이 되어 분류 10개 클래스에 대응한다.

실제 출력 shape가 예상과 다르다면 보통 `padding` 또는 `MaxPool2d` 연산에서 공간 크기 계산이 달라진 경우이며, 이 CNN에서는 각 연산이 모두 기대한 대로 동작했다.


## 2. Batch 크기 N과 Parameter 수 분석

입력의 나머지 축은 `(1,28,28)`로 유지하고, `model.eval()` 상태에서 `N ∈ {1,8,32}`를 각각 넣어 본다.


In [3]:
cnn.eval()
batch_records = []
valid_shapes = []
for N in [1, 8, 32]:
    x_batch = torch.randn(N, 1, 28, 28)
    with torch.no_grad():
        y_batch = cnn(x_batch)
    param_count = sum(p.numel() for p in cnn.parameters())
    batch_records.append({
        'N': N,
        '최종 출력 shape': str(tuple(y_batch.shape)),
        'Parameter 수': param_count,
    })
    valid_shapes.append(tuple(y_batch.shape) == (N, 10))

batch_df = pd.DataFrame(batch_records)
display(batch_df)
print('모든 출력 shape가 (N, 10)인가?', all(valid_shapes))
print('모든 Parameter 수가 같은가?', len({row['Parameter 수'] for row in batch_records}) == 1)


,N,최종 출력 shape,Parameter 수
0,1,"(1, 10)",1379557
1,8,"(8, 10)",1379557
2,32,"(32, 10)",1379557


모든 출력 shape가 (N, 10)인가? True
모든 Parameter 수가 같은가? True


In [4]:
params_by_layer = []
for layer_name, layer in zip(
    [
        'Conv2d(1,32,3,padding=1)',
        'ReLU()',
        'MaxPool2d(2)',
        'Conv2d(32,64,3,padding=1)',
        'ReLU()',
        'MaxPool2d(2)',
        'Conv2d(64,128,3,padding=1)',
        'ReLU()',
        'MaxPool2d(2,padding=1)',
        'Flatten(start_dim=1)',
        'Linear(2048,625)',
        'ReLU()',
        'Dropout()',
        'Linear(625,10)',
    ],
    cnn,
):
    param_count = sum(p.numel() for p in layer.parameters())
    params_by_layer.append({
        'Layer': layer_name,
        'Parameter 수': param_count,
        '학습 가능한 Parameter 유무': '있음' if param_count > 0 else '없음',
    })

param_df = pd.DataFrame(params_by_layer)
display(param_df)


,Layer,Parameter 수,학습 가능한 Parameter 유무
0,"Conv2d(1,32,3,padding=1)",320,있음
1,ReLU(),0,없음
2,MaxPool2d(2),0,없음
3,"Conv2d(32,64,3,padding=1)",18496,있음
4,ReLU(),0,없음
5,MaxPool2d(2),0,없음
6,"Conv2d(64,128,3,padding=1)",73856,있음
7,ReLU(),0,없음
8,"MaxPool2d(2,padding=1)",0,없음
9,Flatten(start_dim=1),0,없음


### 비교 결과

- `N = 1, 8, 32` 모두 최종 출력 shape는 `(N, 10)`으로 동일하다.
- `sum(p.numel() for p in model.parameters())` 값은 모델이 동일하므로 세 행에서 항상 같다.
- 학습 가능한 파라미터는 보통 `Conv2d`와 `Linear` 계층에만 존재하고, `ReLU`, `MaxPool`, `Dropout`, `Flatten`에는 없다.


## 선택 실습: 입력 영상 크기와 Linear 입력 크기가 맞지 않는 경우

아래 코드는 영상 크기가 입력에 따라 `Flatten` 결과와 `Linear`의 `in_features`가 맞지 않을 때 어떤 오류가 발생하는지를 확인한다.


In [5]:
small = nn.Sequential(
    nn.Conv2d(1, 4, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Conv2d(4, 8, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(8 * 7 * 7, 10),
)

for side in [32, 28]:
    debug_image = torch.randn(1, 1, side, side)
    with torch.no_grad():
        debug_features = small[:-1](debug_image)
        print('Input:', debug_image.shape,
              'Flatten:', debug_features.shape,
              'Linear in_features:', small[-1].in_features)
        try:
            print('Output:', small(debug_image).shape)
        except (RuntimeError, ValueError) as error:
            print(type(error).__name__, str(error))


Input: torch.Size([1, 1, 32, 32]) Flatten: torch.Size([1, 512]) Linear in_features: 392
RuntimeError addmm: mat1 and mat2 shapes cannot be multiplied (1x512 and 392x10)
Input: torch.Size([1, 1, 28, 28]) Flatten: torch.Size([1, 392]) Linear in_features: 392
Output: torch.Size([1, 10])


| 입력 영상 크기 | Flatten shape | Linear in_features | 실행 결과 |
|---|---:|---:|---|
| 32 | (1, 512) | 392 | 실패: `Linear`는 392개를 기대하지만 512개가 들어와서 오류 발생 |
| 28 | (1, 392) | 392 | 성공: 출력 shape = `(1, 10)` |

- `32x32` 입력은 2번의 `MaxPool2d(2)` 이후 `8x8` 크기까지 줄어들어 Flatten 수가 512개가 된다.
- `28x28` 입력은 2번의 `MaxPool2d(2)` 이후 `7x7`이 되어 Flatten 수가 392개가 되고, `Linear(392, 10)`과 정확히 맞아떨어진다.
- 즉, `Linear`가 기대하는 입력 수와 일치하지 않으면 `matmul` 오류가 발생한다.
